# Model Data Collection (Stage 0-3)

This notebook initializes model environments and prepares training datasets from model lists.

Forecast horizon for each model is taken from the model list CSV column `output_range`. For example, if a row has `output_range = 36`, data preparation will use a 36-point forecast horizon for that model.

Scope:
- Stage 0: configuration and readiness checks
- Stage 2: data collection (API or stub)
- Stage 3: preprocessing and parquet export

Outputs per model:
- `datasets/train_snapshot.parquet`
- `datasets/validation_snapshot.parquet`
- `datasets/test_snapshot.parquet`
- `reports/quality_summary.csv`
- `reports/dataset_profile.csv`
- `environment/model_environment.json`
- `export_manifest.json`

In [37]:
from __future__ import annotations

import json
import shlex
import subprocess
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

In [38]:
def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ml-server').exists() and (candidate / 'local' / 'models').exists():
            return candidate
    raise FileNotFoundError('Cannot locate repository root containing ml-server and local/models')

REPO_ROOT = _find_repo_root(Path.cwd().resolve())

@dataclass
class BatchCollectionConfig:
    project_root: Path = REPO_ROOT / 'local' / 'models'
    local_dir: Path = project_root / 'training_workspace' / 'models_enabled'
    models_csv: Path = local_dir / 'models_list.csv'
    inputs_csv: Path = local_dir / 'models_inputs.csv'
    output_root: Path = project_root
    horizon_key: str | None = None  # day | month | year | None
    source_mode: str = 'stub'  # auto | api | stub
    api_url: str | None = None
    limit: int | None = 5
    train_ratio: float = 0.7
    validation_ratio: float = 0.15
    timeout_sec: int = 30
    retries: int = 2

cfg = BatchCollectionConfig(source_mode='stub')

print('repo_root  =', REPO_ROOT)
print('models_csv =', cfg.models_csv)
print('inputs_csv =', cfg.inputs_csv)
cfg

repo_root  = /Users/rustamkrikbayev/Documents/projects/forecast
models_csv = /Users/rustamkrikbayev/Documents/projects/forecast/local/models/training_workspace/models_enabled/models_list.csv
inputs_csv = /Users/rustamkrikbayev/Documents/projects/forecast/local/models/training_workspace/models_enabled/models_inputs.csv


BatchCollectionConfig(project_root=PosixPath('/Users/rustamkrikbayev/Documents/projects/forecast/local/models'), local_dir=PosixPath('/Users/rustamkrikbayev/Documents/projects/forecast/local/models/training_workspace/models_enabled'), models_csv=PosixPath('/Users/rustamkrikbayev/Documents/projects/forecast/local/models/training_workspace/models_enabled/models_list.csv'), inputs_csv=PosixPath('/Users/rustamkrikbayev/Documents/projects/forecast/local/models/training_workspace/models_enabled/models_inputs.csv'), output_root=PosixPath('/Users/rustamkrikbayev/Documents/projects/forecast/local/models'), horizon_key=None, source_mode='stub', api_url=None, limit=5, train_ratio=0.7, validation_ratio=0.15, timeout_sec=30, retries=2)

In [39]:
def read_models_csv(path: Path) -> pd.DataFrame:
    probe = pd.read_csv(path, nrows=1)
    return pd.read_csv(path, sep=';') if len(probe.columns) <= 2 else pd.read_csv(path)

models_df = read_models_csv(cfg.models_csv)
inputs_df = pd.read_csv(cfg.inputs_csv, sep=None, engine='python')

preview_columns = [
    column
    for column in ['object_ref', 'onbject', 'type', 'horizont key', 'input_range', 'output_range', 'step', 'model_path']
    if column in models_df.columns
    ]

display(models_df[preview_columns].head(3))
display(inputs_df.head(3))

print('models_rows=', len(models_df))
print('inputs_rows=', len(inputs_df))
print('inputs_types=', inputs_df['input_type'].dropna().unique().tolist())
if 'output_range' in models_df.columns:
    output_ranges = pd.to_numeric(models_df['output_range'], errors='coerce').dropna().astype(int)
    print('output_ranges=', sorted(output_ranges.unique().tolist()))

,object_ref,onbject,type,horizont key,input_range,output_range,step
0,/root/FP/PROJECT/AKMOLA/VostVet_VES/@models/P_...,VostVet_VES,power,day,360,36,3600
1,/root/FP/PROJECT/AKMOLA/@regions/North Kazakhs...,@regions,power,day,360,36,3600
2,/root/FP/PROJECT/AKMOLA/@regions/Kokshetau/loa...,@regions,power,day,360,36,3600


,#,object_ref,description,region,object,type,input_type,input_number,input_ref
0,1156,/root/FP/PROJECT/AKMOLA/VostVet_VES/@models/P_...,ВЭС Восток Ветер+Alcor+Тургай; power,AKMOLA,VostVet_VES,power,historical,1.0,/root/FP/PROJECT/AKMOLA/VostVet_VES/Pgen_sum/a...
1,1214,/root/FP/PROJECT/AKMOLA/@regions/North Kazakhs...,Нагрузка Северо-Казахстанской области;power,AKMOLA,@regions,power,historical,1.0,/root/FP/PROJECT/AKMOLA/@regions/SevKaz/Load/P...
2,1217,/root/FP/PROJECT/AKMOLA/@regions/Kokshetau/loa...,Нагрузка Кокшетауского энергоузла;power,AKMOLA,@regions,power,historical,1.0,/root/FP/PROJECT/AKMOLA/@regions/KOKSHETAU/Loa...


models_rows= 3
inputs_rows= 3
inputs_types= ['historical']
output_ranges= [36]


In [40]:
# Validate output_range in selected models CSV before running collection
if 'output_range' not in models_df.columns:
    raise ValueError('models_csv must contain output_range column')

output_range_values = pd.to_numeric(models_df['output_range'], errors='coerce')
invalid_output_range = models_df[output_range_values.isna() | (output_range_values <= 0)].copy()

if not invalid_output_range.empty:
    preview_invalid = [
        column
        for column in ['object_ref', 'onbject', 'type', 'horizont key', 'output_range']
        if column in invalid_output_range.columns
    ]
    display(invalid_output_range[preview_invalid].head(20))
    raise ValueError(f'Found {len(invalid_output_range)} model(s) with invalid output_range. output_range must be > 0 for all models.')

output_range_series = output_range_values.astype(int)
print('output_range validation: OK')
print('output_range_min=', int(output_range_series.min()))
print('output_range_max=', int(output_range_series.max()))
print('output_range_unique=', sorted(output_range_series.unique().tolist()))

output_range validation: OK
output_range_min= 36
output_range_max= 36
output_range_unique= [36]


In [41]:
script_path = cfg.project_root / 'scripts' / 'batch_model_data_collection.py'
cmd = [
    'python', str(script_path),
    '--models-csv', str(cfg.models_csv),
    '--inputs-csv', str(cfg.inputs_csv),
    '--output-root', str(cfg.output_root),
    '--source-mode', str(cfg.source_mode),
    '--train-ratio', str(cfg.train_ratio),
    '--validation-ratio', str(cfg.validation_ratio),
    '--timeout', str(cfg.timeout_sec),
    '--retries', str(cfg.retries),
]

if cfg.api_url:
    cmd.extend(['--api-url', cfg.api_url])
if cfg.horizon_key:
    cmd.extend(['--horizon-key', cfg.horizon_key])
if cfg.limit is not None:
    cmd.extend(['--limit', str(cfg.limit)])

print('Per-model forecast horizon is taken from models_csv.output_range')
print('Running command:')
print(' '.join(shlex.quote(part) for part in cmd))

proc = subprocess.run(cmd, cwd=str(cfg.project_root), text=True, capture_output=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f'Collection failed with code {proc.returncode}')

Per-model forecast horizon is taken from models_csv.output_range
Running command:
python /Users/rustamkrikbayev/Documents/projects/forecast/local/models/scripts/batch_model_data_collection.py --models-csv /Users/rustamkrikbayev/Documents/projects/forecast/local/models/training_workspace/models_enabled/models_list.csv --inputs-csv /Users/rustamkrikbayev/Documents/projects/forecast/local/models/training_workspace/models_enabled/models_inputs.csv --output-root /Users/rustamkrikbayev/Documents/projects/forecast/local/models --source-mode stub --train-ratio 0.7 --validation-ratio 0.15 --timeout 30 --retries 2 --limit 5



In [42]:
summary_path = cfg.output_root / 'batch_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.exists() else []
summary_df = pd.DataFrame(summary)

display(summary_df.head(10))

if not summary_df.empty and 'status' in summary_df.columns:
    print(summary_df['status'].value_counts(dropna=False))

models_for_summary = read_models_csv(cfg.models_csv)
if 'output_range' in models_for_summary.columns:
    output_range_summary = pd.to_numeric(models_for_summary['output_range'], errors='coerce')
    distribution = output_range_summary.value_counts(dropna=False).sort_index()
    print('\noutput_range distribution (models count):')
    print(distribution.to_string())

    valid_unique = sorted(output_range_summary.dropna().astype(int).unique().tolist())
    if len(valid_unique) > 1:
        print(f'\nMixed horizons detected in models_csv: {valid_unique}')
    elif len(valid_unique) == 1:
        print(f'\nSingle horizon in models_csv: {valid_unique[0]}')
    else:
        print('\nNo valid numeric output_range values found in models_csv')

print('summary_path=', summary_path)

,status,object_ref,output_range,train_snapshot_path,output_dir,rows_total,rows_train,rows_validation,rows_test
0,ok,/root/FP/PROJECT/AKMOLA/VostVet_VES/@models/P_...,36,/Users/rustamkrikbayev/Documents/projects/fore...,/Users/rustamkrikbayev/Documents/projects/fore...,360,251,55,54
1,ok,/root/FP/PROJECT/AKMOLA/@regions/North Kazakhs...,36,/Users/rustamkrikbayev/Documents/projects/fore...,/Users/rustamkrikbayev/Documents/projects/fore...,360,251,55,54
2,ok,/root/FP/PROJECT/AKMOLA/@regions/Kokshetau/loa...,36,/Users/rustamkrikbayev/Documents/projects/fore...,/Users/rustamkrikbayev/Documents/projects/fore...,360,251,55,54


status
ok    3
Name: count, dtype: int64

output_range distribution (models count):
output_range
36    3

Single horizon in models_csv: 36
summary_path= /Users/rustamkrikbayev/Documents/projects/forecast/local/models/batch_summary.json


## Next Step

Use generated parquet snapshots in manual training workflow notebook:
- `docs/training/TRAINING_MANUAL_WORKFLOW.ipynb`

If you need real historical data collection, switch:
- `cfg.source_mode = 'api'`
- `cfg.api_url = 'http://.../history'`